In [7]:
import pandas as pd

file_path = "Banking_Loan_Dataset.xlsx"
customers = pd.read_excel(file_path, sheet_name="Customers")
loans = pd.read_excel(file_path, sheet_name="Loans")
repayments = pd.read_excel(file_path, sheet_name="Repayments")

In [3]:
import pandas as pd

file_path = "Banking_Loan_Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="Customers")

# First 5 rows
print(df.head())

# Number of rows and columns
rows, cols = df.shape
print(f"Rows: {rows}, Columns: {cols}")

# All column names
print(df.columns.tolist())

# Data types
print(df.dtypes)

  customer_id  age  gender       city employment_type  monthly_income  \
0      C00001   40    Male     Indore  Business Owner           22600   
1      C00002   27    Male  Bengaluru        Salaried          115500   
2      C00003   45    Male      Kochi      Government          121500   
3      C00004   46  Female    Chennai        Salaried          185300   
4      C00005   21  Female       Pune        Salaried           94500   

   credit_score account_open_date branch_id          income_band  \
0           699        2024-09-30     BR014           Low (<30K)   
1           676        2023-11-03     BR004         High (>100K)   
2           651        2022-04-02     BR027         High (>100K)   
3           649        2023-08-09     BR016         High (>100K)   
4           786        2018-06-01     BR006  Upper-Mid (60-100K)   

      credit_band  
0  Fair (600-699)  
1  Fair (600-699)  
2  Fair (600-699)  
3  Fair (600-699)  
4  Good (700-799)  
Rows: 10000, Columns: 11
['custo

In [8]:
print(customers.groupby("income_band")["monthly_income"].mean())

income_band
High (>100K)           137980.339463
Low (<30K)              23477.702192
Mid (30-60K)            44668.721840
Upper-Mid (60-100K)     76404.957321
Name: monthly_income, dtype: float64


In [9]:
print(customers.groupby("income_band")["monthly_income"].agg(["mean", "min", "max"]))


                              mean     min     max
income_band                                       
High (>100K)         137980.339463  100100  300000
Low (<30K)            23477.702192   15000   30000
Mid (30-60K)          44668.721840   30100   60000
Upper-Mid (60-100K)   76404.957321   60100  100000


In [10]:
loans["loan_amount_lakhs"] = loans["loan_amount"] / 100000
print(loans[["loan_id", "loan_amount", "loan_amount_lakhs"]].head())

   loan_id  loan_amount  loan_amount_lakhs
0  L000001       780000                7.8
1  L000002       300000                3.0
2  L000003       620000                6.2
3  L000004      1470000               14.7
4  L000005       400000                4.0


In [11]:
loan_customer = loans.merge(customers[["customer_id", "city", "income_band"]],
                            on="customer_id", how="inner")
print(loan_customer[["loan_id", "loan_amount", "city", "income_band"]].head(10))

   loan_id  loan_amount        city          income_band
0  L000001       780000   Ahmedabad         Mid (30-60K)
1  L000002       300000      Indore  Upper-Mid (60-100K)
2  L000003       620000      Mumbai           Low (<30K)
3  L000004      1470000       Delhi  Upper-Mid (60-100K)
4  L000005       400000     Kolkata  Upper-Mid (60-100K)
5  L000006      3310000   Ahmedabad           Low (<30K)
6  L000007       480000  Chandigarh         Mid (30-60K)
7  L000008       970000     Lucknow           Low (<30K)
8  L000009       480000   Ahmedabad  Upper-Mid (60-100K)
9  L000010       960000  Chandigarh  Upper-Mid (60-100K)


In [12]:
print(loans["loan_status"].value_counts())

loan_status
Active       12428
Closed        6044
Defaulted     2028
Name: count, dtype: int64


In [13]:
second_highest = loans["loan_amount"].drop_duplicates().nlargest(2).iloc[-1]
print(second_highest)

4990000


In [14]:
total_borrowing = loans.groupby("customer_id")["loan_amount"].sum().reset_index(name="total_borrowing")
high_value = total_borrowing[total_borrowing["total_borrowing"] > 3_000_000] \
    .sort_values("total_borrowing", ascending=False)
print(high_value)

     customer_id  total_borrowing
7949      C09095         19700000
7679      C08776         16690000
4429      C05032         16140000
1605      C01820         15990000
800       C00905         15610000
...          ...              ...
7732      C08839          3010000
3474      C03957          3010000
2217      C02522          3010000
6778      C07726          3010000
3434      C03911          3010000

[3638 rows x 2 columns]


In [15]:
print(repayments.isnull().sum())

# drop completely empty junk columns
repayments = repayments.dropna(axis=1, how="all")

# fill numeric columns with mean, except payment_date
numeric_cols = repayments.select_dtypes(include="number").columns.drop("payment_date")
for col in numeric_cols:
    repayments[col] = repayments[col].fillna(repayments[col].mean())

print(repayments.isnull().sum())

payment_id            0
loan_id               0
installment_no        0
due_date              0
payment_date        233
amount_due            0
amount_paid           0
days_past_due         0
dpd_bucket            0
payment_status        0
Unnamed: 10       70427
233               70427
dtype: int64
payment_id          0
loan_id             0
installment_no      0
due_date            0
payment_date      233
amount_due          0
amount_paid         0
days_past_due       0
dpd_bucket          0
payment_status      0
dtype: int64


In [16]:
repayments["unpaid_amount"] = repayments["amount_due"] - repayments["amount_paid"]
print(repayments.groupby("payment_status")["unpaid_amount"].sum())

payment_status
Late              0
Missed     12000073
On time           0
Partial    29173654
Name: unpaid_amount, dtype: int64


In [17]:
summary = (
    loans.merge(customers[["customer_id", "income_band"]], on="customer_id", how="inner")
    .groupby("income_band")
    .agg(number_of_loans=("loan_id", "count"),
         total_loan_amount=("loan_amount", "sum"),
         average_loan_amount=("loan_amount", "mean"))
    .sort_values("total_loan_amount", ascending=False)
)
print(summary)

                     number_of_loans  total_loan_amount  average_loan_amount
income_band                                                                 
Mid (30-60K)                    8563        10924990000         1.275837e+06
Upper-Mid (60-100K)             6335         8016470000         1.265425e+06
High (>100K)                    2843         3629250000         1.276556e+06
Low (<30K)                      2759         3603300000         1.306017e+06


In [18]:
top10 = customers.sort_values("monthly_income", ascending=False).head(10)
print(top10[["customer_id", "city", "monthly_income"]])

     customer_id        city  monthly_income
6476      C06477   Bengaluru          300000
4075      C04076      Jaipur          300000
271       C00272       Kochi          300000
7379      C07380   Ahmedabad          300000
4478      C04479  Chandigarh          300000
7750      C07751      Jaipur          300000
5466      C05467   Bengaluru          300000
9684      C09685       Delhi          300000
3678      C03679     Chennai          300000
7984      C07985      Indore          299900


In [19]:
high_income = customers.loc[customers["monthly_income"] > 60000,
                            ["customer_id", "city", "monthly_income"]]
print(high_income)

     customer_id        city  monthly_income
1         C00002   Bengaluru          115500
2         C00003       Kochi          121500
3         C00004     Chennai          185300
4         C00005        Pune           94500
5         C00006     Lucknow           68900
...          ...         ...             ...
9987      C09988     Kolkata           69100
9988      C09989   Ahmedabad           79900
9989      C09990  Chandigarh           80600
9994      C09995     Kolkata           63100
9998      C09999     Chennai           81300

[4460 rows x 3 columns]


In [20]:
print("Duplicate customer records:", customers.duplicated(subset="customer_id").sum())

print("Rows before:", len(customers))
customers = customers.drop_duplicates(subset="customer_id", keep="first").reset_index(drop=True)
print("Rows after:", len(customers))

Duplicate customer records: 0
Rows before: 10000
Rows after: 10000


In [21]:
print(customers.isnull().sum())

customer_id          0
age                  0
gender               0
city                 0
employment_type      0
monthly_income       0
credit_score         0
account_open_date    0
branch_id            0
income_band          0
credit_band          0
dtype: int64
